# Audiobook Pipeline

Convierte un EPUB en audiolibro (M4B con capítulos) con voz personalizada en castellano.

**Runtime:** funciona con T4 GPU (rápido) o sin acelerador (más lento pero sin cuota). Selecciona en Runtime → Change runtime type.

## 1. Instalar dependencias

In [ ]:
!apt-get -qq install ffmpeg > /dev/null
!pip install -q ebooklib beautifulsoup4 lxml piper-tts
!git clone --depth 1 https://github.com/IAHispano/Applio.git /content/Applio 2>/dev/null || echo 'Applio ya clonado'
%cd /content/Applio

# Quitar el sufijo +cu128 de torch en el requirements (funciona tanto en CPU como en GPU)
!sed -i 's/+cu128//g' requirements.txt

# Detectar si hay GPU disponible y elegir índice de torch acorde
import subprocess
has_gpu = subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0

if has_gpu:
    print('GPU detectada, instalando torch...')
    !pip install -q torch==2.7.1 torchaudio==2.7.1 torchvision==0.22.1
else:
    print('Sin GPU, instalando torch CPU...')
    !pip install -q torch==2.7.1 torchaudio==2.7.1 torchvision==0.22.1 --index-url https://download.pytorch.org/whl/cpu

!pip install -q -r requirements.txt 2>&1 | tail -3

## 2. Descargar modelos auxiliares (HuBERT, RMVPE, FCPE, contentvec, Piper)

In [ ]:
import os
os.makedirs('/content/Applio/rvc/models/embedders/contentvec', exist_ok=True)
os.makedirs('/content/Applio/rvc/models/predictors', exist_ok=True)
os.makedirs('/content/piper_model', exist_ok=True)

!curl -sL -o /content/Applio/rvc/models/embedders/hubert_base.pt https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/hubert_base.pt
!curl -sL -o /content/Applio/rvc/models/embedders/contentvec/pytorch_model.bin https://huggingface.co/IAHispano/Applio/resolve/main/Resources/embedders/contentvec/pytorch_model.bin
!curl -sL -o /content/Applio/rvc/models/embedders/contentvec/config.json https://huggingface.co/IAHispano/Applio/resolve/main/Resources/embedders/contentvec/config.json
!curl -sL -o /content/Applio/rvc/models/predictors/rmvpe.pt https://huggingface.co/IAHispano/Applio/resolve/main/Resources/predictors/rmvpe.pt
!curl -sL -o /content/Applio/rvc/models/predictors/fcpe.pt https://huggingface.co/IAHispano/Applio/resolve/main/Resources/predictors/fcpe.pt

!curl -sL -o /content/piper_model/es_ES-davefx-medium.onnx https://huggingface.co/rhasspy/piper-voices/resolve/main/es/es_ES/davefx/medium/es_ES-davefx-medium.onnx
!curl -sL -o /content/piper_model/es_ES-davefx-medium.onnx.json https://huggingface.co/rhasspy/piper-voices/resolve/main/es/es_ES/davefx/medium/es_ES-davefx-medium.onnx.json

print('Modelos descargados')

## 3. Conectar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Configurar rutas

Sube el epub al panel de archivos de Colab (icono de carpeta a la izquierda). Ajusta `EPUB_PATH` con el nombre real del archivo.

In [ ]:
EPUB_PATH   = '/content/libro.epub'
MODEL_PTH   = '/content/drive/MyDrive/audiobook-models/es-male-01.pth'
MODEL_INDEX = '/content/drive/MyDrive/audiobook-models/es-male-01.index'
OUTPUT_M4B  = '/content/drive/MyDrive/audiolibro.m4b'

PIPER_MODEL = '/content/piper_model/es_ES-davefx-medium.onnx'

import os
assert os.path.exists(EPUB_PATH), f'No existe: {EPUB_PATH}'
assert os.path.exists(MODEL_PTH), f'No existe: {MODEL_PTH}'
print('Todo OK')

## 5. Generar el audiolibro

In [ ]:
import os, sys, re, tempfile, subprocess, wave
from pathlib import Path
import ebooklib
from ebooklib import epub
from bs4 import BeautifulSoup

sys.path.insert(0, '/content/Applio')
os.chdir('/content/Applio')

def extract_chapters(epub_path):
    book = epub.read_epub(epub_path)
    chapters = []
    for item in book.get_items_of_type(ebooklib.ITEM_DOCUMENT):
        soup = BeautifulSoup(item.get_content(), 'html.parser')
        for tag in soup(['script','style']): tag.decompose()
        title_tag = soup.find(['h1','h2'])
        title = title_tag.get_text(strip=True) if title_tag else f'Capítulo {len(chapters)+1}'
        text = re.sub(r'\s+',' ', soup.get_text(separator=' ', strip=True)).strip()
        if len(text) > 100:
            chapters.append((title, text))
    return chapters

def piper_tts(text, wav_path):
    r = subprocess.run(['piper','--model', PIPER_MODEL, '--output_file', str(wav_path)],
                       input=text, capture_output=True, text=True)
    if r.returncode != 0: raise RuntimeError(r.stderr)

from rvc.infer.infer import VoiceConverter
vc = VoiceConverter()

def rvc_convert(input_wav, output_wav):
    vc.convert_audio(
        audio_input_path=str(input_wav), audio_output_path=str(output_wav),
        model_path=MODEL_PTH, index_path=MODEL_INDEX if os.path.exists(MODEL_INDEX) else '',
        pitch=0, f0_method='rmvpe', index_rate=0.75, hop_length=128,
        split_audio=False, autotune=False, clean_audio=False, clean_strength=0.7,
        export_format='WAV', embedder_model='contentvec', embedder_model_custom=None,
        upscale_audio=False, resample_sr=0, post_process=False,
    )

def wav_duration_ms(p):
    with wave.open(str(p),'r') as f:
        return int(f.getnframes()/f.getframerate()*1000)

chapters = extract_chapters(EPUB_PATH)
print(f'{len(chapters)} capítulos detectados')

tmp = Path(tempfile.mkdtemp())
rvc_wavs = []
for i,(title,text) in enumerate(chapters):
    print(f'[{i+1}/{len(chapters)}] {title[:60]}')
    pwav = tmp / f'piper_{i:03d}.wav'
    rwav = tmp / f'rvc_{i:03d}.wav'
    piper_tts(text, pwav)
    rvc_convert(pwav, rwav)
    rvc_wavs.append(rwav)

concat = tmp / 'list.txt'
with open(concat,'w') as f:
    for w in rvc_wavs: f.write(f"file '{w}'\n")
combined = tmp / 'combined.wav'
subprocess.run(['ffmpeg','-y','-f','concat','-safe','0','-i',str(concat),'-c','copy',str(combined)], check=True, capture_output=True)

meta = tmp / 'meta.txt'
pos = 0
with open(meta,'w',encoding='utf-8') as f:
    f.write(';FFMETADATA1\n')
    for w,(title,_) in zip(rvc_wavs, chapters):
        dur = wav_duration_ms(w)
        f.write(f'\n[CHAPTER]\nTIMEBASE=1/1000\nSTART={pos}\nEND={pos+dur}\ntitle={title}\n')
        pos += dur

subprocess.run(['ffmpeg','-y','-i',str(combined),'-i',str(meta),
                '-map_metadata','1','-c:a','aac','-b:a','64k','-movflags','+faststart',
                OUTPUT_M4B], check=True, capture_output=True)

print(f'Listo: {OUTPUT_M4B}')

## 6. Descargar el resultado

In [ ]:
from google.colab import files
files.download(OUTPUT_M4B)